#### Import Required Libraries

In [2]:
import os
import pandas as pd
import numpy as np
import re
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import inspect


#### Set up Input URLs and Output Folder Paths

In [3]:
raw_folder_path = os.path.join("..", "Data", "Raw Data", "DOF")
data_folder_path = os.path.abspath(os.path.join(os.getcwd(),"..", "..", "DATA", "DOF"))
AGOL_folder_path = os.path.abspath(os.path.join(os.getcwd(),"..", "..", "AGOL_DATA", "DOF"))
AGOL_resource_folder = os.path.abspath(os.path.join(os.getcwd(),"..", "_"))
image_folder_path = os.path.abspath(os.path.join(os.getcwd(),"..", "..","DATA", "VIS", "DOF"))
#image_output_filename = os.path.join(image_folder_path, "DOF_PCT.jpg")

if not os.path.exists(data_folder_path):
    os.makedirs(data_folder_path)
    
if not os.path.exists(AGOL_folder_path):
    os.makedirs(AGOL_folder_path)
    
if not os.path.exists(image_folder_path):
    os.makedirs(image_folder_path)

if not os.path.exists(AGOL_folder_path):
    os.makedirs(image_folder_path)

urls = [
    'https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx',
    'https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2010-2020by_Geo_Internet.xlsx',
    'https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2023-Geo-InternetVersion.xlsx'
]




### Data Restructure of URLs

#### 2000 DF

In [4]:
def restructure_spreadsheet_2000(url):
    dfs = []

    timestamp_matches = re.findall(r'\d+', url)
    if timestamp_matches:
        timestamp = int(timestamp_matches[0])
    else:
        timestamp = None

    df = pd.read_excel(url, sheet_name=1, skiprows=1, header=None)

    df.iloc[0, :] = df.iloc[0, :].fillna(method='ffill')
    df.iloc[1, :] = df.iloc[1, :].fillna('')

    headers = []
    for category, column in zip(df.iloc[0, :], df.iloc[1, :]):
        if 'Total' in column:
            headers.append((category + ' ' + column).strip())
        else:
            headers.append(column.strip())
    df.columns = headers
    
    df = df.iloc[2:]

    df['Date'] = pd.DatetimeIndex(df['Date']).year
    df = df.dropna(subset=['Date'])
    df['Date'] = df['Date'].astype(int)

    # Initialize 'County' and 'City' columns
    df['County'] = np.nan
    df['City'] = np.nan

    county_temp = None

    # Reset index
    df.reset_index(drop=True, inplace=True)

    for i in range(len(df)):
        if pd.notnull(df.loc[i, 'County / City']):
            if county_temp is not None:
                df.loc[i, 'County'] = county_temp
                df.loc[i, 'City'] = df.loc[i, 'County / City']
                county_temp = None
            else:
                county_temp = df.loc[i, 'County / City']

    df['County'].fillna(method='ffill', inplace=True)
    df['City'].fillna(method='ffill', inplace=True)

    # shift up and then fill the last row
    df['County'] = df['County'].shift(-1)
    df['City'] = df['City'].shift(-1)
    df['County'].fillna(method='ffill', inplace=True)
    df['City'].fillna(method='ffill', inplace=True)

    df = df.drop(columns=['County / City'])

    # Create 'Census Benchmark' column
    df['Census Benchmark'] = np.where(df['Date'].eq(df['Date'].shift()), 'Yes', 'No')

    df = df[['Date', 'County', 'City', 'Census Benchmark'] + [col for col in df.columns if col not in ['Date', 'City', 'County', 'Census Benchmark']]]

    df['SF'] = df['Single'] + df['Mobile Homes']
    df['MF'] = df['Multiple']
    df.drop('Multiple', axis = 1, inplace = True)

    return df


#### 2010 - present DF

In [5]:
def restructure_spreadsheet_2010(url):
    dfs = []

    timestamp = int(re.findall(r'\d+', url)[0])

    df = pd.read_excel(url, sheet_name=1, skiprows=1, header=None)

    df.columns = df.iloc[1, :]
    df = df.iloc[2:]

    df['Date'] = pd.DatetimeIndex(df['Date']).year
    df = df.dropna(subset=['Date'])
    df['Date'] = df['Date'].astype(int)

    # get the column names as a list
    col_names = df.columns.tolist()

    # initialize a counter
    total_counter = 0

    # loop through column names
    for index, name in enumerate(col_names):
        # check if 'Total' is in column name
        if 'Total' in name:
            total_counter += 1  # increment counter
            # rename the first and second occurrence
            if total_counter == 1:
                col_names[index] = 'POPULATION Total'
            elif total_counter == 2:
                col_names[index] = 'HOUSING UNITS Total'

    # update the DataFrame columns
    df.columns = col_names

    # Create 'Census Benchmark' column
    df['Census Benchmark'] = np.where(df['Date'].eq(df['Date'].shift()), 'Yes', 'No')

    df = df[['Date', 'County', 'City', 'Census Benchmark'] + [col for col in df.columns if col not in ['Date', 'City', 'County', 'Census Benchmark']]]
    df['SF'] = df['Single Attached'] + df['Single Detached'] + df['Mobile Homes']
    df['MF'] = df['Two to Four'] + df['Five Plus']
    df.rename(columns={'Persons per Household': 'Persons Per Household'}, inplace=True)

    return df


### Reprocessing/Cleanup/Combining 2000 and 2010-Present DFs

In [6]:
def final_data_processing(df):
    cols_to_keep = [
        'Date', 
        'County', 
        'City',
        'POPULATION Total',
        'Household',
        'Group Quarters', 
        'HOUSING UNITS Total',
        'SF',
        'MF',
        'Occupied',
        'Vacancy Rate',
        'Persons Per Household',
        'Census Benchmark'
    ]
    
    cols_to_drop = df.columns.difference(cols_to_keep)
    df = df.drop(cols_to_drop, axis=1)
    # Trim leading/trailing spaces
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    #df.rename(columns={'County': 'COUNTY1', 'City': 'CITY1'}, inplace=True)
    
    # complete this block according to your needs
    for col in df.columns:
        if df[col].dtype == 'object':
            pass  # Fill in your code here
            
    df.sort_values(by=['County', 'City', 'Date'], inplace=True)
    #df['HOUSING UNITS Total'].fillna(df['SF'] + df['MF'], inplace=True)
    
    # Include the actual mapping here
    county_to_mpo = {
    "Alameda": "MTC",
    "Alpine": "Rest of CA",
    "Amador": "Rest of CA",
    "Butte": "Rest of CA",
    "Calaveras": "Rest of CA",
    "Colusa": "Rest of CA",
    "Contra Costa": "MTC",
    "Del Norte": "Rest of CA",
    "El Dorado": "SACOG",
    "Fresno": "SJ VALLEY",
    "Glenn": "Rest of CA",
    "Humboldt": "Rest of CA",
    "Imperial": "SCAG",
    "Inyo": "Rest of CA",
    "Kern": "SJ VALLEY",
    "Kings": "SJ VALLEY",
    "Lake": "Rest of CA",
    "Lassen": "Rest of CA",
    "Los Angeles": "SCAG",
    "Madera": "SJ VALLEY",
    "Marin": "MTC",
    "Mariposa": "Rest of CA",
    "Mendocino": "Rest of CA",
    "Merced": "SJ VALLEY",
    "Modoc": "Rest of CA",
    "Mono": "Rest of CA",
    "Monterey": "Rest of CA",
    "Napa": "MTC",
    "Nevada": "Rest of CA",
    "Orange": "SCAG",
    "Placer": "SACOG",
    "Plumas": "Rest of CA",
    "Riverside": "SCAG",
    "Sacramento": "SACOG",
    "San Benito": "Rest of CA",
    "San Bernardino": "SCAG",
    "San Diego": "SANDAG",
    "San Francisco": "MTC",
    "San Joaquin": "SJ VALLEY",
    "San Luis Obispo": "Rest of CA",
    "San Mateo": "MTC",
    "Santa Barbara": "Rest of CA",
    "Santa Clara": "MTC",
    "Santa Cruz": "Rest of CA",
    "Shasta": "Rest of CA",
    "Sierra": "Rest of CA",
    "Siskiyou": "Rest of CA",
    "Solano": "MTC",
    "Sonoma": "MTC",
    "Stanislaus": "SJ VALLEY",
    "Sutter": "SACOG",
    "Tehama": "Rest of CA",
    "Trinity": "Rest of CA",
    "Tulare": "Rest of CA",
    "Tuolumne": "Rest of CA",
    "Ventura": "SCAG",
    "Yolo": "SACOG",
    "Yuba": "SACOG",
    }
    
    df['MPO'] = df['County'].map(county_to_mpo)
    df['MPO'].fillna('Rest of CA', inplace=True)
    
    return df
    

### Defining URLs and Running Functions

#### Bringing it all together

In [7]:
def process_url(url):
    all_dataframes = []

    # Check if the URL contains a 4-digit year
    year_match = re.search(r'\b\d{4}\b', url)

    if year_match:
        # If the URL contains a 4-digit year, use the 2010 processing function
        dataframe = restructure_spreadsheet_2010(url)
    else:
        # If the URL does not contain a 4-digit year, use the 2000 processing function
        dataframe = restructure_spreadsheet_2000(url)

    all_dataframes.append(dataframe)

    # Combine all the dataframes into a single dataframe
    combined_dataframe = pd.concat(all_dataframes, ignore_index=True)
    final_combined_dataframe = final_data_processing(combined_dataframe)

    return final_combined_dataframe


def process_multiple_urls(urls):
    all_dataframes = []

    for url in urls:
        dataframe = process_url(url)
        # Strip leading/trailing spaces from all string columns
        dataframe = dataframe.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        all_dataframes.append(dataframe)

    # Combine all the dataframes into a single dataframe
    combined_dataframe = pd.concat(all_dataframes, ignore_index=True)

    # Split the dataframe into two: one where 'Census Benchmark' is 'Yes' and the other where it's 'No'
    yes_df = combined_dataframe[combined_dataframe['Census Benchmark'] == 'Yes']
    no_df = combined_dataframe[combined_dataframe['Census Benchmark'] == 'No']

    # Combine back the dataframes
    combined_dataframe = pd.concat([yes_df, no_df], ignore_index=True)

    # Drop duplicates in the combined dataframe
    combined_dataframe.drop_duplicates(subset=['Date', 'County', 'City'], keep='first', inplace=True)

    return combined_dataframe




final_combined_dataframe = process_multiple_urls(urls)

In [ ]:
final_combined_dataframe[(final_combined_dataframe['Date'] == 2010) & (final_combined_dataframe['Census Benchmark'] == 'No')]

In [8]:
final_combined_dataframe

,Date,County,City,Census Benchmark,POPULATION Total,Household,Group Quarters,HOUSING UNITS Total,Occupied,Vacancy Rate,Persons Per Household,SF,MF,MPO
0,2010,Alameda,Alameda,Yes,73812,72316,1496,32351,30123,0.068870,2.400691,17174,15177,MTC
1,2010,Alameda,Albany,Yes,18539,15377,3162,6712,6309,0.060042,2.437312,4467,2245,MTC
2,2010,Alameda,Balance of County,Yes,141266,138896,2370,51022,48516,0.049116,2.862891,39565,11457,MTC
3,2010,Alameda,Berkeley,Yes,112580,99731,12849,49454,46029,0.069256,2.166699,23202,26252,MTC
4,2010,Alameda,County Total,Yes,1510271,1469752,40519,581372,544046,0.064203,2.701522,361417,219955,MTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17519,2022,Yuba,Marysville,No,12749,11789,960,5098,4790,0.060416,2.461169,3275,1823,SACOG
17520,2023,Yuba,Marysville,No,12606,11604,1002,5100,4792,0.060392,2.421536,3277,1823,SACOG
17522,2021,Yuba,Wheatland,No,3688,3688,0,1360,1323,0.027206,2.787604,1049,311,SACOG
17523,2022,Yuba,Wheatland,No,3645,3645,0,1360,1323,0.027206,2.755102,1049,311,SACOG


In [10]:
def sacog_county_totals_5yr(df, data_folder_path=None):
    func_name = inspect.currentframe().f_code.co_name

    if data_folder_path is None:
        data_folder_path = os.path.abspath(os.path.join(os.getcwd(),"..", "..", "DATA", "DOF"))

    # Define output file name
    data_output_filename = os.path.join(data_folder_path, f"{func_name}.csv")
    
    # Filter based on MPO, Census Benchmark, City, and specific years
    filtered_df = df[(df['MPO'] == 'SACOG') &
                 (df['Census Benchmark'] != 'Yes') &
                 df['City'].isin(['County Total', 'San Francisco']) &
                 (~df['City'].isin(['Balance of County', 'Incorporated'])) &
                 (df['Date'].isin([2000, 2005, 2010, 2015, 2020, 2023]))]

    # Only keep necessary columns
    filtered_df = filtered_df[['Date','County', 'City', 'POPULATION Total', 'Census Benchmark']]

    # Sort by "Population Total" in descending order and drop duplicates
    filtered_df = filtered_df.sort_values('POPULATION Total', ascending=False).drop_duplicates(subset=['Date', 'County', 'City'], keep='first')
    
    filtered_df.to_csv(data_output_filename, index=False) # Don't forget to exclude the index from being written to the file if you don't need it
    
    
    return filtered_df
    
    
    
    


#sacog_county_totals_5yr(final_combined_dataframe, data_folder_path)
sacog_5yr_df = sacog_county_totals_5yr(final_combined_dataframe)
sacog_5yr_df


#publish_csv_to_gis(os.path.join(data_folder_path, "sacog_county_totals_5yr.csv"), "sacog_county_totals_5yr", 'DOF', "sacog_county_totals_5yr", GIS)

,Date,County,City,POPULATION Total,Census Benchmark
16532,2023,Sacramento,County Total,1572453,No
12196,2020,Sacramento,County Total,1553157,No
12191,2015,Sacramento,County Total,1481641,No
5042,2005,Sacramento,County Total,1350523,No
5037,2000,Sacramento,County Total,1223499,No
16360,2023,Placer,County Total,410305,No
11724,2020,Placer,County Total,399015,No
11719,2015,Placer,County Total,371234,No
4591,2005,Placer,County Total,307710,No
4586,2000,Placer,County Total,248399,No


In [19]:
df = sacog_5yr_df.copy()



# Sort by 'County' and 'Date'
df = df.sort_values(by=['County', 'Date'])

# Calculate the difference between consecutive rows for 'POPULATION Total' and 'Date'
df['Population_Diff'] = df.groupby('County')['POPULATION Total'].diff()
df['Year_Diff'] = df.groupby('County')['Date'].diff()

# Avoid NaN errors by filling with appropriate values
df['Year_Diff'].fillna(1, inplace=True)  # Assuming the difference of one year when NaN, adjust if needed
df['Population_Diff'].fillna(0, inplace=True)

# Previous 'POPULATION Total' after shifting
df['Previous_POP'] = df.groupby('County')['POPULATION Total'].shift(1)

# Apply the formula
df['Result'] = ((df['Population_Diff']) / df['Previous_POP']) * 100 / df['Year_Diff']

# Resulting dataframe
df_final = df[['Date', 'County', 'POPULATION Total', 'Result']]
df_final['Result'] = df['Result'].fillna('-')
df_final
#df_final.to_csv('SACOG 5yr Growth.csv')

,Date,County,POPULATION Total,Result
1451,2000,El Dorado,156299,-
1456,2005,El Dorado,171739,1.9757
8584,2015,El Dorado,182530,0.628337
8589,2020,El Dorado,193519,1.204076
15220,2023,El Dorado,189006,-0.777357
4586,2000,Placer,248399,-
4591,2005,Placer,307710,4.775462
11719,2015,Placer,371234,2.064411
11724,2020,Placer,399015,1.496684
16360,2023,Placer,410305,0.943156
